In [1]:
cd "C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph"

C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph


## EMBEDDING 

#### Creating embedding.py file 

## Testing Embedding File

In [22]:
!pytest tests/test_embedding_model.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- C:\Users\Lenovo\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph
configfile: pytest.ini
plugins: anyio-4.10.0, langsmith-0.10.15
collecting ... collected 2 items

tests/test_embedding_model.py::test_embedding_model_load PASSED          [ 50%]
tests/test_embedding_model.py::test_embedding_dimension PASSED           [100%]

============================= 2 passed in 49.96s ==============================


# Vector Store & Retrieval

##  Load Embedding Model

In [23]:
from pathlib import Path

from src.config.config import load_config
from src.data_ingestion.loader import TranscriptLoader
from src.preprocessing.cleaner import TranscriptCleaner
from src.preprocessing.chunker import TranscriptChunker
from src.vector_store.embedding_model import EmbeddingModel

In [27]:
#prepare nodes
config=load_config()

TRANSCRIPT_PATH = (Path(config["paths"]["raw_data"]) /"Meeting_Transcript.txt")


document = TranscriptLoader.load_document(str(TRANSCRIPT_PATH))
clean_document=TranscriptCleaner.clean(document)
nodes=TranscriptChunker.create_nodes(clean_document)

len(nodes)

2026-09-07 15:54:54 | INFO | src.data_ingestion.loader | Transcript loaded successfully: Meeting_Transcript.txt
2026-09-07 15:54:54 | INFO | src.preprocessing.cleaner | Transcript Normalized Sucessfully
2026-09-07 15:54:56 | INFO | src.preprocessing.chunker | Created 2 transcript chunks.


2

In [28]:
#load embedding model
embedding_model=EmbeddingModel.load_model()
embedding_model

2026-09-07 15:57:25,415 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-07 15:57:25,438 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-09-07 15:57:25,721 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-07 15:57:25,726 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-07 15:57:25,748 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-09-07 15:57:25,755 - INFO - Loading SentenceTransformer model from BAAI/b

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-09-07 15:57:28,585 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-07 15:57:28,815 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-07 15:57:29,099 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-07 15:57:29,405 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-07 15:57:29,712 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-07 15:57:29,728 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5', embed_batch_size=10, callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x0000027787682990>, num_workers=None, embeddings_cache=None, rate_limiter=None, max_length=512, normalize=True, query_instruction=None, text_instruction=None, cache_folder=None, show_progress_bar=False)

In [29]:
#lets generate embedding of 1 chunk 
embedding=embedding_model.get_text_embedding(nodes[0].text)

In [30]:
len(embedding)

384

In [33]:
embedding[:10]

[-0.07228470593690872,
 0.021206285804510117,
 -0.020377391949295998,
 -0.06238453462719917,
 -0.04287869855761528,
 -0.030352558940649033,
 0.018756980076432228,
 0.029272591695189476,
 0.0009841292630881071,
 -0.012493474408984184]

In [36]:
#verify multiple chunks

embeddings=[embedding_model.get_text_embedding(node.text) for node in nodes[:3]]
len(embedding),len(embeddings[0])

(384, 384)

# FAISS Vector Index

#### Testing FAISSSTORE INDEX

#### Import FAISS Manager

In [51]:
from src.vector_store.faiss_index import FAISSIndexManager

2026-09-07 16:49:39,119 - INFO - Loading faiss with AVX2 support.
2026-09-07 16:49:39,122 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-09-07 16:49:39,123 - INFO - Loading faiss.
2026-09-07 16:49:39,353 - INFO - Successfully loaded faiss.


In [52]:
vector_index = FAISSIndexManager.create_index(
    nodes=nodes,
    embedding_model=embedding_model,
)

type(vector_index)

2026-09-07 16:50:00 | INFO | src.vector_store.faiss_index | FAISS vector index created successfully.


llama_index.core.indices.vector_store.base.VectorStoreIndex

In [53]:
vector_index

In [54]:
len(nodes)

2

# RETRIVER

### Creating Retriver.py file 

### Testing retiver.py 

In [61]:
!pytest tests/test_retriever.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- C:\Users\Lenovo\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph
configfile: pytest.ini
plugins: anyio-4.10.0, langsmith-0.10.15
collecting ... collected 3 items

tests/test_retriever.py::test_create_retriever PASSED                    [ 33%]
tests/test_retriever.py::test_retrieve_returns_results PASSED            [ 66%]
tests/test_retriever.py::test_retrieved_node_contains_text PASSED        [100%]

======================== 3 passed in 79.01s (0:01:19) =========================


#### Import Retriever

In [62]:
from src.vector_store.retriever import TranscriptRetriever

#### Create Retriever

In [63]:
retriever = TranscriptRetriever.create_retriever(vector_index)

retriever

2026-09-07 19:12:50 | INFO | src.vector_store.retriever | Retriever created with Top-K = 3


#### Asking a Sample Query

In [65]:
query = "What are the main action items discussed?"

results = TranscriptRetriever.retrieve(
    retriever=retriever,
    query=query,
)

len(results)
                             

2

In [66]:
for index, node in enumerate(results, start=1):
    print(f"\n----- Result {index} -----\n")
    print(node.text)


----- Result 1 -----

Product Manager: Good. Let's also inform the management team about this issue.

Customer Support Lead: I will send a message to the support team so they can update customers that we are working on a fix.

Mobile Developer: If the issue is in the login module, I should be able to push a fix by tomorrow.

QA Tester: Once the fix is ready, I will perform regression testing.

Backend Developer: I will also test the authentication endpoints after the fix is deployed.

Product Manager: Great. Please document the issue and the resolution steps.

QA Tester: I will update the issue tracker with all findings.

Customer Support Lead: Should we also prepare a communication message for users?

Product Manager: Yes, we should notify users once the fix is deployed.

Mobile Developer: I will also add better error handling so the app does not crash even if the API fails.

Backend Developer: That would definitely improve stability.

QA Tester: After testing, we should release the 

In [67]:
for result in results:
    print(result.score)

0.5588437531200144
0.5272307338402212
